# IHDP-100

This notebook assesses whether TFM-based base learners improve causal machine learning performance within meta-learning frameworks, compared to conventional baseline methods.

**Dataset**: IHDP (Infant Health and Development Program), a semi-synthetic benchmark derived from a real RCT with simulated potential outcomes.
- 100 replications from the NPCI benchmark (`ihdp_npci_1-100.train.npz` / `.test.npz`)
- 672 train / 75 test samples per replication, 25 covariates
- Outcome = simulated continuous response; true ITE available from `mu0`/`mu1`

**Evaluation protocol**:
- Pre-split train/test sets per replication (fixed NPCI splits)
- All models trained on training set only, evaluated on held-out test set
- R-learner (NonParamDML) and DR-learner (DRLearner) use 5-fold cross-fitting within the training data for nuisance estimation
- LightGBM hyperparameters tuned once per replication via RandomizedSearchCV (30 iterations, 3-fold CV)

**Metrics** (all computed on the test set per replication):
- **PEHE**: Precision in Estimating Heterogeneous Effects — RMSE between predicted and true ITE. Lower is better.
- **ATE Error**: Absolute difference between mean predicted ITE and mean true ITE. Lower is better.

**Models**:
- Meta-learners: S, T, X, R (NonParamDML, cv=5), DR (DRLearner, cv=5)
- Base models: LinearRegression, LightGBM (tuned), TabPFN, TabICL
- Standalone: CausalForestDML (LightGBM nuisance), CausalPFN

## 1. Setup and Imports

In [3]:
# ── IMPORTS ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
import matplotlib.pyplot as plt
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
from sklearn.metrics import mean_squared_error
import warnings
import torch
import tabpfn
from causalpfn import CATEEstimator
from collections import defaultdict
import time


# Ensure we use the latest version of TabPFN.
# We use TabPFN 2.6, which corresponds to GitHub 7.x versions
print(tabpfn.__version__)


# ── DEVICE DETECTION ─────────────────────────────────────────────────────────
# Detect available device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Using device: {device}")

# CausalPFN device: CUDA or CPU
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL device
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")


# ── EXTRA ─────────────────────────────────────────────────────────
np.random.seed(42) # Set random seed for reproducibility
# Suppress warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message=".*feature names.*")

import os
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning" # Ignore warnings
os.environ["TABPFN_NO_TELEMETRY_PROMPT"] = "1" # Skip TabPFN prompt

# To prevent crashes. Avoid CPU over-subscription.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7.1.1
Using device: mps
CausalPFN device: cpu
TabICL device: cpu


In [4]:
# ── Import TabPFN 2.5 alongside 2.6 ────────────────────────────────────────
import subprocess, sys as _sys

TABPFN25_DIR = "./tabpfn_v25_install"
os.makedirs(TABPFN25_DIR, exist_ok=True)

# Install tabpfn 2.5 to isolated directory if not already present
_tabpfn25_installed = any("tabpfn" in d for d in os.listdir(TABPFN25_DIR))
if not _tabpfn25_installed:
    print("Installing TabPFN 2.5 to isolated directory...")
    subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "tabpfn==6.4.1",
        f"--target={TABPFN25_DIR}", "--quiet", "--no-deps"
    ])
    print("Done.")

# Save current tabpfn 2.6 module references
_tabpfn26_mods = {k: v for k, v in _sys.modules.items()
                  if k == "tabpfn" or k.startswith("tabpfn.")}

# Temporarily inject 2.5 path and load its classes
_sys.path.insert(0, TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]

import tabpfn as _tabpfn25
TabPFNRegressor25 = _tabpfn25.TabPFNRegressor
TabPFNClassifier25 = _tabpfn25.TabPFNClassifier
TABPFN25_VERSION = _tabpfn25.__version__

# Restore tabpfn 2.6
_sys.path.remove(TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]
_sys.modules.update(_tabpfn26_mods)

print(f"TabPFN 2.5 version: {TABPFN25_VERSION}")
print(f"TabPFN 2.6 version: {tabpfn.__version__}")

TabPFN 2.5 version: 6.4.1
TabPFN 2.6 version: 7.1.1


## 2. Define Models, Tuning, and Metrics

We define LightGBM hyperparameter tuning functions (using `RandomizedSearchCV`) and evaluation metrics. LightGBM is tuned **once per replication** — the best hyperparameters are then reused across all meta-learners for that replication.

In [5]:
# ── TUNING ─────────────────────────────────────────────────────────
# LightGBM hyperparameter search space
LGBM_GRID = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [20, 50, 100],
    'n_estimators':      [200, 500, 1000],
}

N_ITER = 30    # RandomizedSearchCV draws
N_EST       = 1000  # base tree count (overridden by grid)
NUISANCE_CV = 5    # K-fold cross-fitting for R- and DR-learner nuisance models

# LightGBM Tuning
def tune_lgbm(X, y, classifier = False, stratify = None, n_iter = N_ITER, seed = 42):
    """Tune LGBM via RandomizedSearchCV. Returns best_params_ dict.

    - classifier = False  → LGBMRegressor, scored by neg_MSE
    - classifier = True   → LGBMClassifier, scored by neg_log_loss
    - stratify          → use StratifiedKFold on this variable (regression only);
                          if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)

    if classifier:
        # Propensity Model (classification)
        base    = LGBMClassifier(n_estimators = N_EST, random_state = seed, verbose = -1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle = True, random_state = seed).split(X, y))
    else:
        # Outcome Model (regression): S-, R-, and DR-learner
        base    = LGBMRegressor(n_estimators = N_EST, random_state = seed, verbose = -1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle = True, random_state = seed).split(X, stratify))
        else:
            # Single Treatment Arm: T- and X-learner
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle = True, random_state = seed).split(X))

    search = RandomizedSearchCV(
        base, LGBM_GRID, n_iter = n_iter, scoring = scoring,
        cv = cv, n_jobs = -1, random_state = seed,
    )

    search.fit(X, y)
    return search.best_params_

# Pseudo-Outcome Tuning: X-, R-, and DR-learner final-stage models
def make_lgbm_final(seed = 42):
    """LGBM wrapped in RandomizedSearchCV for final-stage tuning on pseudo-outcomes."""
    return RandomizedSearchCV(
        LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1),
        LGBM_GRID, n_iter=N_ITER, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=seed,
    )



# ── Evaluation Metric functions ─────────────────────────────────────────────────────────
def calculate_pehe(predicted_ite, true_ite):
    return np.sqrt(mean_squared_error(true_ite, predicted_ite))

def calculate_ate_error(predicted_ite, true_ite):
    return np.abs(predicted_ite.mean() - true_ite.mean())


## 3. Data Loading (IHDP 1-100)

We load 100 realizations of the IHDP dataset from local `.npz` files and store them for evaluation.

In [8]:
# ── Data Loading ─────────────────────────────────────────────────────────
# Store results for all runs
# Structure: results[meta_learner][base_model][metric] = list of values
all_results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Load IHDP data from local npz files (100 replications)
train_data = np.load('ihdp_npci_1-100.train.npz')
test_data = np.load('ihdp_npci_1-100.test.npz')

n_datasets = 100
processed_datasets = []
feature_names = [f"x{j}" for j in range(25)]

print(f"Loading {n_datasets} replications from local IHDP files...")

# Process Datasets
for i in range(n_datasets):
    X_train = pd.DataFrame(train_data['x'][:, :, i], columns=feature_names)
    T_train, Y_train, mu0_train, mu1_train = (train_data[k][:, i] for k in ('t', 'yf', 'mu0', 'mu1'))

    X_test = pd.DataFrame(test_data['x'][:, :, i], columns=feature_names)
    T_test, Y_test, mu0_test, mu1_test = (test_data[k][:, i] for k in ('t', 'yf', 'mu0', 'mu1'))

    true_ITE_test = mu1_test - mu0_test

    processed_datasets.append({
        'id': i + 1,
        'X_train': X_train,
        'X_test': X_test,
        'T_train': T_train,
        'T_test': T_test,
        'Y_train': Y_train,
        'Y_test': Y_test,
        'true_ITE_test': true_ITE_test
    })


# ── Data Summary ─────────────────────────────────────────────────────────
rep = processed_datasets[0]
X_tr, T_tr, Y_tr = rep['X_train'], rep['T_train'], rep['Y_train']
X_te, T_te, Y_te = rep['X_test'],  rep['T_test'],  rep['Y_test']

n_train, n_test = len(T_tr), len(T_te)
n_total    = n_train + n_test
n_treated  = int(T_tr.sum()) + int(T_te.sum())
n_control  = n_total - n_treated
ate_naive  = Y_tr[T_tr == 1].mean() - Y_tr[T_tr == 0].mean()
true_ate   = rep['true_ITE_test'].mean()

Y_all_rep  = np.concatenate([Y_tr, Y_te])

print("=" * 55)
print("IHDP Dataset Summary (replication 1)")
print("=" * 55)
print(f"  Replications   : {n_datasets}")
print(f"  Train samples  : {n_train}  |  Test samples: {n_test}")
print(f"  Total (rep 1)  : {n_total}")
print(f"  Treated        : {n_treated}  ({100*n_treated/n_total:.1f}%)")
print(f"  Control        : {n_control}  ({100*n_control/n_total:.1f}%)")
print(f"  Features       : {X_tr.shape[1]}")
print(f"  Outcome (Y)    : range [{Y_all_rep.min():.2f}, {Y_all_rep.max():.2f}]")
print(f"                   mean  {Y_all_rep.mean():.2f}  (std {Y_all_rep.std():.2f})")
print(f"  ATE naive      : {ate_naive:+.4f}  (treated mean - control mean, train)")
print(f"  True ATE       : {true_ate:+.4f}  (mean true ITE on test set)")
print()
print("Outcome by treatment arm (train, rep 1):")
print(f"  Treated  mean Y: {Y_tr[T_tr==1].mean():.4f}  (std {Y_tr[T_tr==1].std():.4f})")
print(f"  Control  mean Y: {Y_tr[T_tr==0].mean():.4f}  (std {Y_tr[T_tr==0].std():.4f})")

print(f"\nLoaded {len(processed_datasets)} replications.")
print(f"Train size: {X_tr.shape[0]}, Test size: {X_te.shape[0]}, Features: {X_tr.shape[1]}")

Loading 100 replications from local IHDP files...
IHDP Dataset Summary (replication 1)
  Replications   : 100
  Train samples  : 672  |  Test samples: 75
  Total (rep 1)  : 747
  Treated        : 139  (18.6%)
  Control        : 608  (81.4%)
  Features       : 25
  Outcome (Y)    : range [-1.54, 11.27]
                   mean  3.16  (std 2.18)
  ATE naive      : +4.0047  (treated mean - control mean, train)
  True ATE       : +4.0563  (mean true ITE on test set)

Outcome by treatment arm (train, rep 1):
  Treated  mean Y: 6.4408  (std 1.1264)
  Control  mean Y: 2.4361  (std 1.6095)

Loaded 100 replications.
Train size: 672, Test size: 75, Features: 25


## 4. Unified Meta-Learner Evaluation

We evaluate all five meta-learners (S, T, X, R, DR) in a single loop over the 100 IHDP replications. For each replication, we tune LightGBM once for the outcome model (regressor) and once for the propensity model (classifier), then reuse those tuned models across all meta-learners. This avoids redundant tuning and ensures consistency.

In [ ]:
# ── Evaluation Prep ─────────────────────────────────────────────────────────
print("Evaluating all meta-learners across 100 IHDP replications...\n")

for dataset in processed_datasets:
    i = dataset['id']
    n = len(processed_datasets)
    print(f"\n{'='*70}")
    print(f" Dataset {i}/{n}")
    print(f"{'='*70}")

    X_train, X_test = dataset['X_train'], dataset['X_test']
    T_train         = dataset['T_train']
    Y_train         = dataset['Y_train']
    true_ITE_test   = dataset['true_ITE_test']


    # ── Arm splits (needed for T- and X-learner tuning) ──────────────────────
    ctrl = T_train == 0
    trt  = T_train == 1
    X_ctrl, Y_ctrl = X_train[ctrl], Y_train[ctrl]
    X_trt,  Y_trt  = X_train[trt],  Y_train[trt]


    # ── LightGBM hyperparameter tuning ───────────────────────────────────────
    t0 = time.time()
    X_with_T        = np.column_stack([X_train, T_train])
    params_s        = tune_lgbm(X_with_T, Y_train, stratify = T_train)  # S-learner outcome (X+T features)
    params_outcome  = tune_lgbm(X_train,  Y_train, stratify = T_train)  # outcome nuisance (R-learner model_y)
    params_prop     = tune_lgbm(X_train,  T_train, classifier = True)   # propensity model
    params_ctrl     = tune_lgbm(X_ctrl,   Y_ctrl)                       # T/X control arm outcome
    params_trt      = tune_lgbm(X_trt,    Y_trt)                        # T/X treated arm outcome
    print(f"  LightGBM tuning: {time.time() - t0:.1f}s")


    # ── Model configs ────────────────────────────────────────────────────────
    # Local shorthands to avoid repeating constructor args across 4 models × 11 roles
    def _lgbm_r(params, seed = 42): return LGBMRegressor(random_state=seed, verbose=-1, **params)
    def _lgbm_c(params, seed = 42): return LGBMClassifier(random_state=seed, verbose=-1, **params)
    def _tabpfn_r():                return TabPFNRegressor(device=device)
    def _tabpfn_c():                return TabPFNClassifier(device=device)
    def _tabicl_r(seed = 42):       return TabICLRegressor(device=tabicl_device, random_state=seed, verbose=False)
    def _tabicl_c(seed = 42):       return TabICLClassifier(device=tabicl_device, random_state = seed, verbose=False)
    def _tabpfn25_r():              return TabPFNRegressor25(device=device)
    def _tabpfn25_c():              return TabPFNClassifier25(device=device)

    base_model_configs = {
        'LinearRegression': {
            's_model':       LinearRegression(),
            't_models':      (LinearRegression(), LinearRegression()),
            'x_models':      (LinearRegression(), LinearRegression()),
            'x_cate':        (LinearRegression(), LinearRegression()),
            'x_propensity':  LogisticRegression(max_iter=1000, random_state=42),
            'r_model_y':     LinearRegression(),
            'r_model_t':     LogisticRegression(max_iter=1000, random_state=42),
            'r_model_final': LinearRegression(),
            'dr_regression': LinearRegression(),
            'dr_propensity': LogisticRegression(max_iter=1000, random_state=42),
            'dr_final':      LinearRegression(),
        },
        'LightGBM': {
            's_model':       _lgbm_r(params_s),
            't_models':      (_lgbm_r(params_ctrl, 42), _lgbm_r(params_trt, 43)),
            'x_models':      (_lgbm_r(params_ctrl, 42), _lgbm_r(params_trt, 43)),
            'x_cate':        (make_lgbm_final(46), make_lgbm_final(47)),
            'x_propensity':  _lgbm_c(params_prop, 44),
            'r_model_y':     _lgbm_r(params_outcome, 42),
            'r_model_t':     _lgbm_c(params_prop, 43),
            'r_model_final': make_lgbm_final(44),
            'dr_regression': _lgbm_r(params_s, 42),       # tuned on [X,T], matches DRLearner's internal input
            'dr_propensity': _lgbm_c(params_prop, 44),
            'dr_final':      make_lgbm_final(45),
        },
        'TabPFN_v2.5': {
            's_model':       _tabpfn25_r(),
            't_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_cate':        (_tabpfn25_r(), _tabpfn25_r()),
            'x_propensity':  _tabpfn25_c(),
            'r_model_y':     _tabpfn25_r(),
            'r_model_t':     _tabpfn25_c(),
            'r_model_final': make_lgbm_final(44),
            'dr_regression': _tabpfn25_r(),
            'dr_propensity': _tabpfn25_c(),
            'dr_final':      _tabpfn25_r(),
        },
        'TabPFN_v2.6': {
            's_model':       _tabpfn_r(),
            't_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_cate':        (_tabpfn_r(), _tabpfn_r()),
            'x_propensity':  _tabpfn_c(),
            'r_model_y':     _tabpfn_r(),
            'r_model_t':     _tabpfn_c(),
            'r_model_final': make_lgbm_final(44),  # TabPFN not suited for residual-on-residual
            'dr_regression': _tabpfn_r(),
            'dr_propensity': _tabpfn_c(),
            'dr_final':      _tabpfn_r(),
        },
        'TabICL': {
            's_model':       _tabicl_r(),
            't_models':      (_tabicl_r(42), _tabicl_r(43)),
            'x_models':      (_tabicl_r(42), _tabicl_r(43)),
            'x_cate':        (_tabicl_r(44), _tabicl_r(45)),
            'x_propensity':  _tabicl_c(),
            'r_model_y':     _tabicl_r(),
            'r_model_t':     _tabicl_c(),
            'r_model_final': make_lgbm_final(44),  # TabICL not suited for residual-on-residual
            'dr_regression': _tabicl_r(),
            'dr_propensity': _tabicl_c(),
            'dr_final':      _tabicl_r(),
        }
    }

    # ── Evaluation ─────────────────────────────────────────────────────────
    for name, cfg in base_model_configs.items():
        
        # ── S-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        s_learner = SLearner(overall_model = cfg['s_model'])
        s_learner.fit(Y_train, T_train, X = X_train)
        te = s_learner.effect(X_test)
        
        all_results['S'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['S'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  S-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── T-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        t_learner = TLearner(models = cfg['t_models'])
        t_learner.fit(Y_train, T_train, X = X_train)
        te = t_learner.effect(X_test)

        all_results['T'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['T'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  T-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── X-Learner ─────────────────────────────────────────────────────────
        t0 = time.time()
        x_learner = XLearner(models = cfg['x_models'], cate_models = cfg['x_cate'], propensity_model = cfg['x_propensity'])
        x_learner.fit(Y_train, T_train, X = X_train)
        te = x_learner.effect(X_test)

        all_results['X'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['X'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  X-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── R-learner (NonParamDML) ────────────────────────────────────────────
        t0 = time.time()
        r_learner = NonParamDML(
            model_y = cfg['r_model_y'], model_t = cfg['r_model_t'],
            model_final = cfg['r_model_final'], discrete_treatment = True, cv = NUISANCE_CV
        )
        r_learner.fit(Y_train, T_train, X = X_train)
        te = r_learner.effect(X_test)

        all_results['R'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['R'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  R-learner  + {name:20s}: {time.time() - t0:.1f}s")


        # ── DR-learner ────────────────────────────────────────────
        t0 = time.time()
        dr_learner = DRLearner(
            model_regression = cfg['dr_regression'],
            model_propensity = cfg['dr_propensity'],
            model_final = cfg['dr_final'],
            min_propensity = 0.05,  # prevents extreme IPW weights
            cv = NUISANCE_CV
        )
        dr_learner.fit(Y_train, T_train, X=X_train)
        te = dr_learner.effect(X_test)
        
        all_results['DR'][name]['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['DR'][name]['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  DR-learner + {name:20s}: {time.time() - t0:.1f}s")


    # ── Causal Forest (CausalForestDML) ───────────────────────────────────────
    try:
        t0 = time.time()
        cf = CausalForestDML(
            model_y = LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            model_t = LGBMClassifier(random_state=43, verbose=-1, **params_prop),
            discrete_treatment = True,
            cv=NUISANCE_CV,
            n_estimators = 200,
            min_samples_leaf = 5,
            random_state = 42,
        )
        cf.tune(Y_train, T_train, X=X_train)
        cf.fit(Y_train, T_train, X=X_train)
        te = cf.effect(X_test)

        all_results['CF']['CausalForest']['pehe'].append(calculate_pehe(te, true_ITE_test))
        all_results['CF']['CausalForest']['ate_error'].append(calculate_ate_error(te, true_ITE_test))
        print(f"  CF         + {'CausalForest':20s}: {time.time() - t0:.1f}s")

    except Exception as e:
        print(f"\nCausalForest error on dataset {i}: {e}")


    # ── CausalPFN ───────────────────────────────────────
    try:
        print(f"  [CausalPFN]...", end=" ", flush=True)
        _t0 = time.time()
        cpfn = CATEEstimator(device = causalpfn_device, verbose = False)

        X_cpfn_train = np.asarray(X_train, dtype=np.float32)
        T_cpfn_train = np.asarray(T_train, dtype=np.float32).ravel()
        Y_cpfn_train = np.asarray(Y_train, dtype=np.float32).ravel()
        X_cpfn_test = np.asarray(X_test, dtype=np.float32)


        cpfn.fit(X_cpfn_train, T_cpfn_train, Y_cpfn_train)
        te = cpfn.estimate_cate(X_cpfn_test)

        if "torch" in str(type(te)):
            te = te.detach().cpu().numpy()

        te = np.asarray(te, dtype=np.float32).reshape(-1)

        all_results["CausalPFN"]["CausalPFN"]["pehe"].append(calculate_pehe(te, true_ITE_test))
        all_results["CausalPFN"]["CausalPFN"]["ate_error"].append(calculate_ate_error(te, true_ITE_test))
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalPFN error on dataset {i}: {exc}")

print("\nAll meta-learner evaluations complete.")


Evaluating all meta-learners across 100 IHDP replications...


 Dataset 1/100


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


  LightGBM tuning: 226.2s
  S-learner  + LinearRegression    : 0.0s
  T-learner  + LinearRegression    : 0.0s
  X-learner  + LinearRegression    : 0.0s
  R-learner  + LinearRegression    : 0.0s
  DR-learner + LinearRegression    : 0.0s
  S-learner  + LightGBM            : 0.4s
  T-learner  + LightGBM            : 0.6s
  X-learner  + LightGBM            : 55.0s
  R-learner  + LightGBM            : 58.3s
  DR-learner + LightGBM            : 58.3s
  S-learner  + TabPFN_v2.5         : 5.8s
  T-learner  + TabPFN_v2.5         : 5.7s
  X-learner  + TabPFN_v2.5         : 22.4s
  R-learner  + TabPFN_v2.5         : 126.1s
  DR-learner + TabPFN_v2.5         : 106.7s
  S-learner  + TabPFN_v2.6         : 6.8s
  T-learner  + TabPFN_v2.6         : 6.3s
  X-learner  + TabPFN_v2.6         : 24.6s
  R-learner  + TabPFN_v2.6         : 134.1s
  DR-learner + TabPFN_v2.6         : 121.6s
  S-learner  + TabICL              : 9.6s
  T-learner  + TabICL              : 9.5s
  X-learner  + TabICL              : 

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


  LightGBM tuning: 221.4s
  S-learner  + LinearRegression    : 0.0s
  T-learner  + LinearRegression    : 0.0s
  X-learner  + LinearRegression    : 0.1s
  R-learner  + LinearRegression    : 0.0s
  DR-learner + LinearRegression    : 0.0s
  S-learner  + LightGBM            : 0.4s
  T-learner  + LightGBM            : 0.4s
  X-learner  + LightGBM            : 54.2s
  R-learner  + LightGBM            : 57.7s
  DR-learner + LightGBM            : 59.0s
  S-learner  + TabPFN_v2.5         : 5.8s
  T-learner  + TabPFN_v2.5         : 6.0s
  X-learner  + TabPFN_v2.5         : 22.3s
  R-learner  + TabPFN_v2.5         : 126.3s
  DR-learner + TabPFN_v2.5         : 107.1s
  S-learner  + TabPFN_v2.6         : 6.8s
  T-learner  + TabPFN_v2.6         : 6.4s
  X-learner  + TabPFN_v2.6         : 24.7s
  R-learner  + TabPFN_v2.6         : 133.2s
  DR-learner + TabPFN_v2.6         : 125.4s
  S-learner  + TabICL              : 9.7s
  T-learner  + TabICL              : 9.5s
  X-learner  + TabICL              : 

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


  LightGBM tuning: 224.8s
  S-learner  + LinearRegression    : 0.0s
  T-learner  + LinearRegression    : 0.0s
  X-learner  + LinearRegression    : 0.0s
  R-learner  + LinearRegression    : 0.0s
  DR-learner + LinearRegression    : 0.0s
  S-learner  + LightGBM            : 0.4s
  T-learner  + LightGBM            : 0.3s
  X-learner  + LightGBM            : 57.8s


## 5. Aggregated Results

We report the Mean and Standard Error of PEHE and ATE Error across the 100 replications.

In [ ]:
# ── Results ─────────────────────────────────────────────────────────
summary_rows = []

for meta in ['S', 'T', 'X', 'R', 'DR', 'CF', 'CausalPFN']:
    if meta == 'CausalPFN':
        models = ['CausalPFN']
    elif meta == 'CF':
        models = ['CausalForest']
    else:
        base_models = ['LinearRegression', 'LightGBM', 'TabPFN_v2.6', 'TabPFN_v2.5', 'TabICL']
        models = base_models

    for model in models:
        pehes = all_results[meta][model]['pehe']
        ate_errs = all_results[meta][model]['ate_error']

        if not pehes:
            continue

        n = len(pehes)
        summary_rows.append({
            'Meta-Learner': meta,
            'Base Model': model,
            'PEHE Mean': np.mean(pehes),
            'PEHE SE': np.std(pehes) / np.sqrt(n),
            'ATE Error Mean': np.mean(ate_errs),
            'ATE Error SE': np.std(ate_errs) / np.sqrt(n)
        })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# Export to CSV
csv_path = 'benchmark_results_IHDP.csv'
df_summary.to_csv(csv_path, index=False)
print(f"\nResults exported to {csv_path}")

import seaborn as sns
import matplotlib.pyplot as plt

# Style (optional but nice)
sns.set(style="whitegrid")

if not df_summary.empty:
    df_meta = df_summary[~df_summary['Meta-Learner'].isin(['CausalPFN', 'CF'])]
    df_standalone = df_summary[df_summary['Meta-Learner'].isin(['CausalPFN', 'CF'])]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    metrics = ['PEHE', 'ATE Error']

    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        # ── BARPLOT WITH ERROR BARS ───────────────────────────────
        if not df_meta.empty:
            sns.barplot(
                data=df_meta,
                x='Base Model',
                y=f'{metric} Mean',
                hue='Meta-Learner',
                ax=ax,
                ci=None ,
                palette = 'Blues')

            # Manually add error bars
            for i, row in df_meta.iterrows():
                x_pos = list(df_meta['Base Model'].unique()).index(row['Base Model'])
                hue_offset = list(df_meta['Meta-Learner'].unique()).index(row['Meta-Learner'])

                # adjust position slightly per hue (Seaborn offset)
                n_hue = df_meta['Meta-Learner'].nunique()
                total_width = 0.8
                width = total_width / n_hue
                offset = (hue_offset - (n_hue - 1) / 2) * width

                ax.errorbar(
                    x=x_pos + offset,
                    y=row[f'{metric} Mean'],
                    yerr=row[f'{metric} SE'],
                    fmt='none',
                    ecolor='black',
                    capsize=4,
                    linewidth=1
                )

        # ── HORIZONTAL REFERENCE LINES ────────────────────────────
        for _, row in df_standalone.iterrows():
            val = row[f'{metric} Mean']

            if row['Meta-Learner'] == 'CausalPFN':
                color = 'darkred'
            elif row['Meta-Learner'] == 'CF':
                color = 'darkblue'
            else:
                color = 'darkred'

            ax.axhline(
                y=val,
                linestyle='--',
                linewidth=1.5,
                color=color,
                label=f"{row['Meta-Learner']}-{row['Base Model']} ({val:.3f})"
            )

        # ── STYLING ───────────────────────────────────────────────
        ax.set_title(f'{metric} (Mean ± SE)')
        ax.set_ylabel(metric)
        ax.set_xlabel('Base Model')
        ax.grid(True, alpha=0.3, axis='y')

        ax.legend(loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.savefig("resultsihdp_plot_seaborn.png", dpi=300, bbox_inches='tight')
    plt.show()

else:
    print("No results to plot.")

# --- Save results to disk ---
import pickle

_save_path = 'benchmark_results_IHDP.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump({
        'all_results': {m: {n: dict(d) for n, d in v.items()} for m, v in all_results.items()},
        'N_ITER': N_ITER,
        'N_EST': N_EST,
        'NUISANCE_CV': NUISANCE_CV,
        'n_datasets': n_datasets,
    }, f)
print(f"Results saved to {_save_path}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ── Setup ─────────────────────────────────────────────────────────────────────
META_ORDER  = ['S', 'T', 'X', 'R', 'DR']
MODEL_ORDER = ['LinearRegression', 'LightGBM', 'TabPFN_v2.5', 'TabPFN_v2.6','TabICL']
MODEL_LABELS = ['LinReg', 'LightGBM',  'TabPFN_v2.5', 'TabPFN_v2.6', 'TabICL']

df_main = df_summary[
    df_summary['Meta-Learner'].isin(META_ORDER) &
    df_summary['Base Model'].isin(MODEL_ORDER)
].copy()

def make_heatmap_data(metric_col, se_col):
    mean_mat = (df_main
                .pivot(index='Meta-Learner', columns='Base Model', values=metric_col)
                .reindex(index=META_ORDER, columns=MODEL_ORDER))
    se_mat   = (df_main
                .pivot(index='Meta-Learner', columns='Base Model', values=se_col)
                .reindex(index=META_ORDER, columns=MODEL_ORDER))
    annot = np.empty(mean_mat.shape, dtype=object)
    for i in range(mean_mat.shape[0]):
        for j in range(mean_mat.shape[1]):
            m, s = mean_mat.iloc[i, j], se_mat.iloc[i, j]
            annot[i, j] = f'{m:.3f}\n±{s:.3f}' if not pd.isna(m) else ''
    mean_mat.columns = MODEL_LABELS
    return mean_mat, annot

pehe_mat, pehe_annot = make_heatmap_data('PEHE Mean',       'PEHE SE')
ate_mat,  ate_annot  = make_heatmap_data('ATE Mean',  'ATE SE')

# ── Reference values ──────────────────────────────────────────────────────────
def _ref(meta, metric):
    row = df_summary[df_summary['Meta-Learner'] == meta]
    return row[metric].values[0] if len(row) else None

refs = {
    'CausalForest': (_ref('CF', 'PEHE Mean'),       _ref('CF', 'ATE Mean')),
    'CausalPFN':    (_ref('CausalPFN', 'PEHE Mean'), _ref('CausalPFN', 'ATE Mean')),
}

# ── Heatmaps ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mat, annot, title, ref_idx in [
    (axes[0], pehe_mat, pehe_annot, 'PEHE  (lower is better)',       0),
    (axes[1], ate_mat,  ate_annot,  'ATE Error  (lower is better)',  1),
]:
    sns.heatmap(mat.astype(float), annot=annot, fmt='', ax=ax,
                cmap='', linewidths=0.5, linecolor='white',
                cbar_kws={'shrink': 0.75, 'label': 'mean'},
                annot_kws={'size': 9})
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel('Base Model', fontsize=10)
    ax.set_ylabel('Meta-Learner', fontsize=10)
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

    ref_text = '   '.join(
        f'{name}: {vals[ref_idx]:.3f}'
        for name, vals in refs.items() if vals[ref_idx] is not None
    )
    if ref_text:
        ax.text(0.5, -0.16, ref_text, transform=ax.transAxes,
                ha='center', fontsize=8, color='dimgray', style='italic')

plt.suptitle('IHDP Benchmark — Meta-Learner × Base Model', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

# ── Ranking table ─────────────────────────────────────────────────────────────
df_rank = df_summary.copy()
df_rank['Model'] = df_rank.apply(
    lambda r: r['Base Model'] if r['Meta-Learner'] in ('CF', 'CausalPFN')
              else f"{r['Meta-Learner']}-{r['Base Model']}", axis=1)
df_rank = (df_rank
           .sort_values('PEHE Mean')
           .reset_index(drop=True))
df_rank.index += 1
df_rank.index.name = 'Rank'

print(df_rank[['Model', 'PEHE Mean', 'PEHE SE', 'ATE Mean', 'ATE SE']]
      .rename(columns={'ATE Mean': 'ATE Mean', 'ATE SE': 'ATE SE'})
      .round(4)
      .to_string())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# load your results directly
df = df_summary.copy()  # replace with whatever your df is called

# create model label combining meta-learner and base model
df['Model'] = df['Meta-Learner'] + '-' + df['Base Model']

# rank by PEHE
df = df.sort_values('PEHE Mean').reset_index(drop=True)
df['Rank'] = df.index + 1

# family color mapping
def get_family(base):
    if 'TabPFN_v2.6' in base: return 'TFM'
    if 'TabPFN_v2.5' in base: return 'TFM'
    if 'TabICL' in base: return 'TFM'
    if 'LightGBM' in base: return 'Tree'
    if 'CausalForest' in base: return 'CausalForest'
    if 'CausalPFN' in base: return 'CausalPFN'
    return 'Linear'

df['Family'] = df['Base Model'].apply(get_family)

family_colors = {
    'TFM':          '#185FA5',
    'Tree':         '#3B6D11',
    'Linear':       '#993C1D',
    'CausalForest': '#BA7517',
    'CausalPFN':    '#7F77DD',
}

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# assumes your df has columns: Meta-Learner, Base Model, AUQC Mean, AUQC SE
# adjust column names to match yours
df = df_summary.copy()

# define order of learners and base models
learner_order = ['S', 'T', 'X', 'DR', 'R']
base_order    = ['LinearRegression', 'LightGBM', 'TabICL', 'TabPFN_v2.6', 'TabPFN_v2.5']

# colors matching your existing palette
colors = {
    'LinearRegression': '#993C1D',  # coral/red
    'LightGBM':         '#3B6D11',  # green
    'TabICL':           '#1D9E75',  # teal
    'TabPFN_v2.6':           '#185FA5',  # blue
    'TabPFN_v2.5':       '#7F77DD',  # purple
}

markers = {
    'LinearRegression': 's',
    'LightGBM':         'D',
    'TabICL':           '^',
    'TabPFN_v2.6':           'o',
    'TabPFN_v2.5':       'o',
}

fig, ax = plt.subplots(figsize=(8, 5))

n_learners   = len(learner_order)
n_bases      = len(base_order)
width        = 0.7  # total width per learner group
offsets      = np.linspace(-width/2, width/2, n_bases)

for bi, base in enumerate(base_order):
    sub = df[df['Base Model'] == base].copy()
    sub = sub.set_index('Meta-Learner')

    xs, ys, ses = [], [], []
    for li, learner in enumerate(learner_order):
        if learner in sub.index:
            xs.append(li + offsets[bi])
            ys.append(sub.loc[learner, 'PEHE Mean'])
            ses.append(sub.loc[learner, 'PEHE SE'])

    ax.errorbar(
        xs, ys, yerr=ses,
        fmt=markers[base],
        color=colors[base],
        markersize=6,
        linewidth=0,
        elinewidth=1.2,
        capsize=3,
        capthick=1,
        label=base,
        zorder=3,
    )

# reference line at zero (random targeting)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--', alpha=0.6)

ax.set_xticks(range(n_learners))
ax.set_xticklabels(learner_order, fontsize=11)
ax.set_xlabel('Meta-learner', fontsize=11)
ax.set_ylabel('AUQC (mean ± SE)', fontsize=11)
ax.set_title('NSW dataset — AUQC by meta-learner and base model\n'
             '(50 splits, higher is better)', fontsize=11, pad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linewidth=0.4, alpha=0.4)

ax.legend(
    title='Base model',
    title_fontsize=9,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
)

plt.tight_layout()
plt.savefig('nsw_auqc_dotplot.pdf', dpi=300, bbox_inches='tight')
plt.show()
# ── latex table ───────────────────────────────────────────────────────────────
def make_latex_table(df):
    lines = []
    lines.append(r'\begin{table}[ht]')
    lines.append(r'\centering')
    lines.append(r'\caption{IHDP results ranked by root-PEHE (mean $\pm$ SE over 100 realizations, lower is better).}')
    lines.append(r'\label{tab:ihdp}')
    lines.append(r'\begin{tabular}{rlcccc}')
    lines.append(r'\toprule')
    lines.append(r'Rank & Model & \multicolumn{2}{c}{Root-PEHE} & \multicolumn{2}{c}{ATE Error} \\')
    lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
    lines.append(r' & & Mean & SE & Mean & SE \\')
    lines.append(r'\midrule')
    for _, row in df.iterrows():
        lines.append(
            rf'{int(row["Rank"])} & {row["Model"]} & '
            rf'{row["PEHE Mean"]:.3f} & {row["PEHE SE"]:.3f} & '
            rf'{row["ATE Error Mean"]:.3f} & {row["ATE Error SE"]:.3f} \\'
        )
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

print(make_latex_table(df))

# 6. Leaderboard